# Multiple Linear Regression
## Housing Case Study

This notebook merges the two existing housing regression notebooks into one clear workflow. It covers the full multiple linear regression process for the housing dataset, including data exploration, encoding, scaling, feature selection, model fitting, residual analysis, and test evaluation.

### What you will learn
- How to inspect and visualise housing data
- How to convert categorical features into numeric form
- How to split and scale data for regression
- How to build and interpret multiple linear regression models
- How to apply feature selection with RFE and VIF
- How to perform residual analysis and evaluate the final model

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.feature_selection import RFE
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

In [ ]:
housing = pd.read_csv('Housing.csv')
housing.head()

In [ ]:
print('Shape:', housing.shape)
print('\nInfo:')
housing.info()

print('Summary statistics:')
housing.describe()

## Dataset Overview
This dataset contains information about houses and the factors that affect their sale price. The target variable is `price`, while other columns describe property features such as size, bedrooms, bathrooms, and amenities.

The most important beginner topics here are:
- `price`: target variable to predict
- `area`, `bedrooms`, `bathrooms`, `stories`, `parking`: numeric predictors
- `mainroad`, `guestroom`, `basement`, `hotwaterheating`, `airconditioning`, `prefarea`: binary `yes/no` features
- `furnishingstatus`: categorical feature with three levels


In [ ]:
print('Columns:')
print(list(housing.columns))
print('\nMissing values per column:')
print(housing.isnull().sum())
print('\nUnique values in categorical columns:')
print(housing[['mainroad', 'guestroom', 'basement', 'hotwaterheating', 'airconditioning', 'prefarea', 'furnishingstatus']].nunique())


## Exploratory Data Analysis

We first look at relationships among numeric features and then inspect categorical variables to understand how they relate to `price`.

In [ ]:
sns.pairplot(housing)
plt.show()

In [ ]:
plt.figure(figsize=(20, 12))
plt.subplot(2, 3, 1)
sns.boxplot(x='mainroad', y='price', data=housing)
plt.subplot(2, 3, 2)
sns.boxplot(x='guestroom', y='price', data=housing)
plt.subplot(2, 3, 3)
sns.boxplot(x='basement', y='price', data=housing)
plt.subplot(2, 3, 4)
sns.boxplot(x='hotwaterheating', y='price', data=housing)
plt.subplot(2, 3, 5)
sns.boxplot(x='airconditioning', y='price', data=housing)
plt.subplot(2, 3, 6)
sns.boxplot(x='furnishingstatus', y='price', data=housing)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))
sns.boxplot(x='furnishingstatus', y='price', hue='airconditioning', data=housing)
plt.title('Price by furnishingstatus and airconditioning')
plt.show()

## Data Preparation

Convert binary categorical features to 0/1 and create dummy variables for the multi-level `furnishingstatus` feature.

In [ ]:
varlist = ['mainroad', 'guestroom', 'basement', 'hotwaterheating', 'airconditioning', 'prefarea']

def binary_map(x):
    return x.map({'yes': 1, 'no': 0})

housing[varlist] = housing[varlist].apply(binary_map)

status = pd.get_dummies(housing['furnishingstatus'], drop_first=True)
housing = pd.concat([housing, status], axis=1)
housing.drop(['furnishingstatus'], axis=1, inplace=True)
housing.head()

## Train-Test Split and Scaling

We use a train-test split to evaluate how the model performs on unseen data. The training set is used to fit the model, and the test set is held back to measure how well predictions generalize.

Scaling is applied to numeric features so the model treats variables on similar scales. This is especially useful when features have very different ranges, such as `area` versus `bathrooms`.

In [ ]:
df_train, df_test = train_test_split(housing, train_size=0.7, test_size=0.3, random_state=100)

num_vars = ['area', 'bedrooms', 'bathrooms', 'stories', 'parking', 'price']
scaler = MinMaxScaler()
df_train[num_vars] = scaler.fit_transform(df_train[num_vars])
df_train.head()

In [ ]:
df_train.describe()

## Correlation and Feature Relationships

A correlation heatmap and scatter plot help identify the strongest predictors for `price`.

In [ ]:
plt.figure(figsize=(16, 10))
sns.heatmap(df_train.corr(), annot=True, cmap='YlGnBu')
plt.title('Correlation Matrix')
plt.show()

In [ ]:
plt.figure(figsize=(6, 6))
plt.scatter(df_train['area'], df_train['price'])
plt.xlabel('area')
plt.ylabel('price')
plt.title('Area vs Price')
plt.show()

## Model Building with Statsmodels

Linear regression assumes a linear relationship between predictors and the target. Before building the model, keep these assumptions in mind:
- linearity: the relationship between features and target should be approximately linear
- independence: observations should be independent of each other
- homoscedasticity: residuals should have constant variance
- normality of errors: residuals should be approximately normal
- no strong multicollinearity: predictors should not be highly correlated

We will start with a simple model using one predictor, then expand to multiple features and compare model statistics.


In [ ]:
y_train = df_train.pop('price')
X_train = df_train

X_train_lm = sm.add_constant(X_train[['area']])
lr = sm.OLS(y_train, X_train_lm).fit()
print(lr.summary())

intercept = lr.params['const']
slope = lr.params['area']
plt.scatter(X_train_lm['area'], y_train)
plt.plot(X_train_lm['area'], intercept + slope * X_train_lm['area'], 'r')
plt.xlabel('area')
plt.ylabel('price')
plt.title('Simple Regression: price ~ area')
plt.show()

In [ ]:
X_train_lm = sm.add_constant(X_train[['area', 'bathrooms', 'bedrooms']])
lr = sm.OLS(y_train, X_train_lm).fit()
print(lr.summary())

## Recursive Feature Elimination (RFE)

Use RFE to select the most important features for the regression model.

In [ ]:
lm = LinearRegression()
lm.fit(X_train, y_train)
rfe = RFE(lm, 10)
rfe = rfe.fit(X_train, y_train)

selected_features = X_train.columns[rfe.support_]
print('Selected features by RFE:')
print(list(selected_features))

In [ ]:
X_train_rfe = X_train[selected_features]
X_train_rfe_const = sm.add_constant(X_train_rfe)
lm_rfe = sm.OLS(y_train, X_train_rfe_const).fit()
print(lm_rfe.summary())

### Check Multicollinearity with VIF

Variance Inflation Factor helps identify features that are highly correlated with each other. Low VIF values are preferred.

In [ ]:
X = X_train_rfe.copy()
vif = pd.DataFrame()
vif['Feature'] = X.columns
vif['VIF'] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
vif['VIF'] = round(vif['VIF'], 2)
vif.sort_values(by='VIF', ascending=False, inplace=True)
vif

## Final Model Selection

Drop features with high p-values or high VIFs and build the final model from the selected predictors.

In [ ]:
X_train_final = X_train_rfe.copy()
if 'bedrooms' in X_train_final.columns:
    X_train_final = X_train_final.drop(['bedrooms'], axis=1)
if 'semi-furnished' in X_train_final.columns:
    X_train_final = X_train_final.drop(['semi-furnished'], axis=1)
if 'basement' in X_train_final.columns:
    X_train_final = X_train_final.drop(['basement'], axis=1)

X_train_final_const = sm.add_constant(X_train_final)
lm_final = sm.OLS(y_train, X_train_final_const).fit()
print(lm_final.summary())

In [ ]:
X_final = X_train_final.copy()
vif_final = pd.DataFrame()
vif_final['Feature'] = X_final.columns
vif_final['VIF'] = [variance_inflation_factor(X_final.values, i) for i in range(X_final.shape[1])]
vif_final['VIF'] = round(vif_final['VIF'], 2)
vif_final.sort_values(by='VIF', ascending=False, inplace=True)
vif_final

## Test Set Predictions and Evaluation

Use the final model to make predictions on the held-out test data and evaluate using standard regression metrics.

Important evaluation metrics:
- R2: proportion of variance in the target explained by the model
- RMSE: root mean squared error, which penalizes larger prediction errors
- MAE: mean absolute error, which reports average absolute prediction error

A higher R2 and lower RMSE/MAE indicate a better fit on the test set.


## Residual Analysis

Check whether the model residuals are approximately normally distributed and whether residuals are approximately constant across fitted values.

In [ ]:
y_train_pred = lm_final.predict(X_train_final_const)
residuals = y_train - y_train_pred
plt.figure(figsize=(8, 5))
sns.histplot(residuals, bins=20, kde=True)
plt.title('Residuals Distribution')
plt.xlabel('Residual')
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(y_train_pred, residuals, alpha=0.7)
plt.axhline(0, color='red', linestyle='--')
plt.xlabel('Fitted values')
plt.ylabel('Residuals')
plt.title('Residuals vs Fitted values')
plt.show()

In [ ]:
df_test[num_vars] = scaler.transform(df_test[num_vars])
y_test = df_test.pop('price')
X_test = df_test.copy()
X_test_final = X_test[X_train_final.columns]
X_test_final_const = sm.add_constant(X_test_final)
y_pred = lm_final.predict(X_test_final_const)

print('R2:', r2_score(y_test, y_pred))
print('RMSE:', np.sqrt(mean_squared_error(y_test, y_pred)))
print('MAE:', mean_absolute_error(y_test, y_pred))

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_pred, alpha=0.7)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
plt.xlabel('Actual price')
plt.ylabel('Predicted price')
plt.title('Actual vs Predicted')
plt.show()

## Conclusion

This merged notebook demonstrates a complete multiple linear regression workflow on the housing dataset. It includes preprocessing of categorical variables, scaling, exploratory analysis, feature selection, model diagnostics, and test evaluation. The final model is built with selected predictors and validated on the held-out test set.

Note: This notebook uses a single train-test split for evaluation. For stronger beginner practice, cross-validation is recommended because it tests model stability across multiple train-test splits and reduces the risk of a lucky or unlucky split.
